# Tutorial 11: Complex Problem Solving

This tutorial explores how ACT-R handles multi-step reasoning, strategy selection, and planning. We'll look at classic problems like Tower of Hanoi and water jugs, then move on to more realistic domains like mathematical problem solving and insight. The goal is to understand how cognitive architectures model the full range of human problem-solving behavior, from systematic planning to creative breakthroughs.

## 1. Introduction to Problem Solving in ACT-R

Complex problem solving involves planning, strategy selection, and managing multiple goals. ACT-R models these through goal stacks and production rules.

In [ ]:
import pyactr as actr
import numpy as np
import time

# Water jug problem: get exactly 4 gallons using 3-gallon and 5-gallon jugs
water_jug_model = actr.ACTRModel()

actr.chunktype("jug_state", "jug3 jug5 goal_amount")
actr.chunktype("operator", "name precondition effect")
actr.chunktype("problem_goal", "state current_state target_amount")

# Possible actions
operators = [
    ("fill3", "jug3<3", "jug3=3"),
    ("fill5", "jug5<5", "jug5=5"),
    ("empty3", "jug3>0", "jug3=0"),
    ("empty5", "jug5>0", "jug5=0"),
    ("pour3to5", "jug3>0,jug5<5", "transfer"),
    ("pour5to3", "jug5>0,jug3<3", "transfer")
]

for name, pre, eff in operators:
    water_jug_model.decmem.add(
        actr.makechunk(
            chunktype="operator",
            name=name,
            precondition=pre,
            effect=eff
        )
    )

water_jug_model.productionstring(
    name="select_operator",
    string='''
    =g>
        isa problem_goal
        state planning
        current_state =state
    ==>
    =g>
        state retrieving_operator
    +retrieval>
        isa operator
'''
)

water_jug_model.productionstring(
    name="apply_operator",
    string='''
    =g>
        isa problem_goal
        state retrieving_operator
    =retrieval>
        isa operator
        name =op
    ==>
    =g>
        state applying
        Applying operator: =op
'''
)

print("Water jug problem solver initialized!")
print("Goal: Get exactly 4 gallons")
print("Tools: 3-gallon jug and 5-gallon jug")

## 2. Tower of Hanoi: Recursive Problem Solving

The Tower of Hanoi demonstrates hierarchical goal decomposition and recursive problem solving.

In [ ]:
import pyactr as actr

hanoi_model = actr.ACTRModel()
hanoi_model.model_parameters['subsymbolic'] = True

actr.chunktype("hanoi_goal", "state disk from_peg to_peg via_peg parent_goal")
actr.chunktype("disk_location", "disk peg")
actr.chunktype("subgoal_link", "parent child")

# Recursive decomposition: to move N disks, first move N-1 disks to spare peg
hanoi_model.productionstring(
    name="decompose_goal",
    string='''
    =g>
        isa hanoi_goal
        state plan
        disk =n
        from_peg =from
        to_peg =to
        via_peg =via
    ==>
    =g>
        state subgoaling
    +subgoal1>
        isa hanoi_goal
        state plan
        disk =n
        from_peg =from
        to_peg =via
        via_peg =to
        parent_goal =g
'''
)

hanoi_model.productionstring(
    name="move_single_disk",
    string='''
    =g>
        isa hanoi_goal
        state plan
        disk 1
        from_peg =from
        to_peg =to
    ==>
    =g>
        state moving
        Move disk 1 from =from to =to
'''
)

hanoi_model.productionstring(
    name="subgoal_return",
    string='''
    =g>
        isa hanoi_goal
        state complete
        parent_goal =parent
    ==>
    =parent>
        state continue
    ~g>
'''
)

goal = actr.makechunk(
    chunktype="hanoi_goal",
    state="plan",
    disk="3",
    from_peg="A",
    to_peg="C",
    via_peg="B"
)
hanoi_model.goal.add(goal)

print("Tower of Hanoi solver (3 disks, optimal = 7 moves)")

## 3. Mathematical Problem Solving

Humans use different strategies to solve algebra problems depending on their experience. Experts tend to use systematic methods, while novices often resort to guess-and-check. This model captures that variability through utility-based strategy selection.

In [ ]:
import pyactr as actr
import random

algebra_model = actr.ACTRModel()
algebra_model.model_parameters['subsymbolic'] = True
algebra_model.model_parameters['utility_noise'] = 0.5

actr.chunktype("equation", "left_side right_side variable")
actr.chunktype("strategy", "name steps utility")
actr.chunktype("solve_goal", "state equation strategy step")

strategies = [
    actr.makechunk(
        chunktype="strategy",
        name="isolate",
        steps="subtract,divide",
        utility="5"
    ),
    actr.makechunk(
        chunktype="strategy",
        name="guess_check",
        steps="guess,evaluate,adjust",
        utility="3"
    ),
    actr.makechunk(
        chunktype="strategy",
        name="backwards",
        steps="reverse_ops",
        utility="4"
    )
]

for strategy in strategies:
    algebra_model.decmem.add(strategy)

algebra_model.productionstring(
    name="select_strategy",
    string='''
    =g>
        isa solve_goal
        state start
        equation =eq
    ==>
    =g>
        state retrieving_strategy
    +retrieval>
        isa strategy
    ''',
    utility=5
)

algebra_model.productionstring(
    name="apply_isolate",
    string='''
    =g>
        isa solve_goal
        state retrieving_strategy
    =retrieval>
        isa strategy
        name isolate
    ==>
    =g>
        state solving
        strategy isolate
        step subtract_constants
        Using systematic isolation strategy
    ''',
    utility=6
)

algebra_model.productionstring(
    name="apply_guess",
    string='''
    =g>
        isa solve_goal
        state retrieving_strategy
    =retrieval>
        isa strategy
        name guess_check
    ==>
    =g>
        state solving
        strategy guess_check
        step make_guess
        Using guess and check strategy
    ''',
    utility=4
)

print("Algebra Problem Solver: 2x + 5 = 13")
print("\nStrategy selection depends on utility + noise")

## 4. Means-Ends Analysis and Planning

Means-ends analysis is one of the fundamental problem-solving strategies. The idea is simple: identify the difference between your current state and goal state, then take actions that reduce that difference. Here we model route planning using this approach.

In [ ]:
import pyactr as actr

planning_model = actr.ACTRModel()
planning_model.model_parameters['subsymbolic'] = True

actr.chunktype("location", "name connections distance_to_goal")
actr.chunktype("route_goal", "state current_loc target_loc path")
actr.chunktype("difference", "type magnitude action")

locations = [
    ("home", "street1,street2", "5"),
    ("street1", "home,mall,street3", "4"),
    ("street2", "home,park", "3"),
    ("street3", "street1,office", "2"),
    ("mall", "street1,office", "3"),
    ("park", "street2,office", "2"),
    ("office", "street3,mall,park", "0")
]

for name, conn, dist in locations:
    planning_model.decmem.add(
        actr.makechunk(
            chunktype="location",
            name=name,
            connections=conn,
            distance_to_goal=dist
        )
    )

planning_model.productionstring(
    name="find_difference",
    string='''
    =g>
        isa route_goal
        state planning
        current_loc =current
        target_loc =target
    ?retrieval>
        buffer empty
    ==>
    =g>
        state analyzing
    +retrieval>
        isa location
        name =current
'''
)

planning_model.productionstring(
    name="select_best_move",
    string='''
    =g>
        isa route_goal
        state analyzing
        current_loc =current
    =retrieval>
        isa location
        connections =connections
        distance_to_goal =dist
    ==>
    =g>
        state moving
        At =current, distance to goal: =dist
        Evaluating connections: =connections
'''
)

planning_model.productionstring(
    name="make_move",
    string='''
    =g>
        isa route_goal
        state moving
        current_loc =current
        path =path
    ==>
    =g>
        state planning
        current_loc =next
        path =path,=next
        Moving to: =next
'''
)

print("Route planning: home to office")
print("Using hill climbing (always reduce distance to goal)")

## 5. Insight and Restructuring

Some problems require us to see them differently before we can solve them. The classic example is the nine-dot problem: connect 9 dots arranged in a grid using 4 straight lines without lifting your pen. Most people fail because they unconsciously assume the lines must stay within the boundary of the dots. When that constraint is relaxed, the solution becomes obvious.

In [ ]:
import pyactr as actr
import random

insight_model = actr.ACTRModel()
insight_model.model_parameters['subsymbolic'] = True
insight_model.model_parameters['spreading_activation'] = True

actr.chunktype("problem_space", "constraint assumption status")
actr.chunktype("insight_goal", "state constraint_active attempts")
actr.chunktype("solution_attempt", "approach success")

insight_model.decmem.add(
    actr.makechunk(
        chunktype="problem_space",
        constraint="stay_inside_dots",
        assumption="implicit",
        status="active"
    )
)

for i in range(5):
    insight_model.decmem.add(
        actr.makechunk(
            chunktype="solution_attempt",
            approach="within_bounds",
            success="false"
        )
    )

insight_model.productionstring(
    name="constrained_attempt",
    string='''
    =g>
        isa insight_goal
        state solving
        constraint_active yes
    ==>
    =g>
        state attempting
        attempts =attempts
        Trying to connect dots staying within bounds...
    ''',
    utility=5
)

insight_model.productionstring(
    name="detect_impasse",
    string='''
    =g>
        isa insight_goal
        state attempting
        attempts =n
    ==>
    =g>
        state impasse
    +retrieval>
        isa problem_space
        status active
        Impasse detected after =n attempts
'''
)

insight_model.productionstring(
    name="relax_constraint",
    string='''
    =g>
        isa insight_goal
        state impasse
    =retrieval>
        isa problem_space
        constraint stay_inside_dots
    ==>
    =g>
        state restructured
        constraint_active no
    =retrieval>
        status relaxed
        AHA! Can go outside the dot boundary!
    ''',
    utility=2
)

insight_model.productionstring(
    name="unconstrained_solve",
    string='''
    =g>
        isa insight_goal
        state restructured
        constraint_active no
    ==>
    =g>
        state solved
        Drawing lines outside the dots...
        Solution found! 4 lines connect all 9 dots.
'''
)

print("Nine-Dot Problem")
print("Repeated failures trigger impasse detection")
print("Impasse leads to constraint relaxation (insight)")

## 6. Analogical Problem Solving

Duncker's radiation problem is a classic demonstration of analogical reasoning. The problem: destroy a tumor with radiation, but high-intensity radiation damages healthy tissue. The solution involves using multiple weak beams from different angles that converge on the tumor. Most people don't solve this spontaneously, but if they first read about a similar military problem (using divided forces to capture a fortress), they transfer the solution.

In [ ]:
import pyactr as actr

analogy_model = actr.ACTRModel()
analogy_model.model_parameters['subsymbolic'] = True
analogy_model.model_parameters['partial_matching'] = True
analogy_model.model_parameters['activation_trace'] = True

actr.chunktype("problem_schema", "type goal obstacle solution domain")
actr.chunktype("current_problem", "description goal obstacle domain")
actr.chunktype("analogy_goal", "state target_problem source_solution")

# Source analogy: military fortress
analogy_model.decmem.add(
    actr.makechunk(
        chunktype="problem_schema",
        type="fortress",
        goal="capture_fortress",
        obstacle="mines_on_roads",
        solution="divide_army_multiple_roads",
        domain="military"
    )
)

analogy_model.decmem.add(
    actr.makechunk(
        chunktype="problem_schema",
        type="irrigation",
        goal="water_all_fields",
        obstacle="main_channel_too_small",
        solution="split_into_channels",
        domain="agriculture"
    )
)

analogy_model.productionstring(
    name="find_analogy",
    string='''
    =g>
        isa analogy_goal
        state searching
        target_problem =problem
    =problem>
        isa current_problem
        goal =goal
        obstacle =obstacle
    ==>
    =g>
        state retrieving
    +retrieval>
        isa problem_schema
        goal =goal
        obstacle =obstacle
'''
)

analogy_model.productionstring(
    name="map_solution",
    string='''
    =g>
        isa analogy_goal
        state retrieving
    =retrieval>
        isa problem_schema
        type =source_type
        solution =solution
        domain =source_domain
    =problem>
        isa current_problem
        domain =target_domain
    ==>
    =g>
        state mapping
        source_solution =solution
        Found analogy from =source_domain domain
        Source solution: =solution
        Mapping to =target_domain domain...
'''
)

target = actr.makechunk(
    chunktype="current_problem",
    description="destroy_tumor_with_radiation",
    goal="destroy_tumor",
    obstacle="high_radiation_damages_tissue",
    domain="medical"
)

print("Radiation Problem")
print("Partial matching allows retrieval based on structural similarity")
print("Solution: multiple weak beams from different angles")

## 7. Complete Problem Solver: Sudoku

Sudoku combines many aspects of human problem solving: constraint satisfaction, strategy selection, and knowing when to escalate to more complex methods. This model uses multiple strategies ordered by difficulty, preferring simpler methods when they work.

In [ ]:
import pyactr as actr
import numpy as np

sudoku_model = actr.ACTRModel()
sudoku_model.model_parameters['subsymbolic'] = True

actr.chunktype("cell", "row col value candidates box")
actr.chunktype("constraint", "type location values")
actr.chunktype("strategy", "name complexity utility")
actr.chunktype("sudoku_goal", "state strategy focus_cell")

strategies = [
    ("naked_single", "1", "10"),
    ("hidden_single", "2", "8"),
    ("pointing_pairs", "3", "6"),
    ("box_line", "4", "5"),
    ("backtrack", "5", "3")
]

for name, complexity, utility in strategies:
    sudoku_model.decmem.add(
        actr.makechunk(
            chunktype="strategy",
            name=name,
            complexity=complexity,
            utility=utility
        )
    )

sudoku_model.productionstring(
    name="select_strategy",
    string='''
    =g>
        isa sudoku_goal
        state scanning
    ==>
    =g>
        state retrieving_strategy
    +retrieval>
        isa strategy
    ''',
    utility=10
)

sudoku_model.productionstring(
    name="apply_naked_single",
    string='''
    =g>
        isa sudoku_goal
        state retrieving_strategy
    =retrieval>
        isa strategy
        name naked_single
    ==>
    =g>
        state finding_singles
        strategy naked_single
    +retrieval>
        isa cell
        value nil
        candidates =c
        Looking for naked singles...
    ''',
    utility=12
)

sudoku_model.productionstring(
    name="place_value",
    string='''
    =g>
        isa sudoku_goal
        state finding_singles
    =retrieval>
        isa cell
        row =r
        col =c
        candidates =val
    ==>
    =g>
        state updating
    =retrieval>
        value =val
        candidates nil
        Placing =val at (=r, =c)
'''
)

sudoku_model.productionstring(
    name="propagate_constraints",
    string='''
    =g>
        isa sudoku_goal
        state updating
    ==>
    =g>
        state propagating
        Updating constraints in affected row, column, and box
'''
)

sudoku_model.productionstring(
    name="try_harder_strategy",
    string='''
    =g>
        isa sudoku_goal
        state stuck
        strategy =current
    ==>
    =g>
        state scanning
    +retrieval>
        isa strategy
        complexity > =current!complexity
        No progress with =current, trying harder strategy...
    ''',
    utility=8
)

print("Sudoku Solver")
print("Strategies: naked singles → hidden singles → pointing pairs → box/line → backtracking")
print("Models human preference for simpler methods")

## Exercises

1. **Cryptarithmetic**: Build a model that solves SEND + MORE = MONEY using constraint satisfaction and backtracking.

2. **Chess tactics**: Model pattern recognition for common tactical motifs (forks, pins, skewers).

3. **Debugging**: Create a model of how programmers generate and test hypotheses about bugs.

4. **Multiple insight types**: Extend the nine-dot model to handle different kinds of constraint relaxation.

5. **Collaborative solving**: Model two agents sharing information and dividing subtasks.

## Summary

This tutorial covered the major phenomena in human problem solving:

- Goal hierarchies and subgoaling (Tower of Hanoi)
- Strategy selection based on utility (algebra solving)
- Means-ends analysis and hill climbing (route planning)
- Constraint relaxation and insight (nine-dot problem)
- Analogical transfer (radiation problem)
- Multi-strategy problem solving (Sudoku)

ACT-R models all of these through the same basic mechanisms: production rules, declarative memory, and subsymbolic activation. The architecture doesn't need separate modules for "creative" vs "analytical" thinking; both emerge from how knowledge is organized and retrieved.